# Resolve AI — Complaint classifier fine-tuning

Fine-tunes **XLM-RoBERTa** to label customer complaints written in **English, Roman Urdu and mixed** text with:

| Task | Labels |
|---|---|
| `category` | refund_request, return_exchange, wrong_item, damaged_item, delivery_issue, payment_issue, cancellation, general_inquiry, other (greetings / thanks / small talk) |
| `urgency` | low, medium, high |
| `sentiment` | negative, neutral, positive |

It also trains a **TF-IDF + Logistic Regression baseline** so the report can compare against it, and reports **macro-F1 per language group** (as required by the evaluation checklist).

### Data
- **Bitext Customer Support dataset** (public, English, downloaded automatically from Hugging Face) — real support messages; its intents are mapped to our categories. It has **no urgency/sentiment labels**, so it is used for the `category` task only.
- **UCI Roman Urdu Data Set** (public, ~20k Roman Urdu sentences, downloaded automatically from Hugging Face) — real Roman Urdu text for the `sentiment` task only.
- **Your CSV** (upload) — English / Roman Urdu / mixed complaints with all three labels. Use `complaint_seed_dataset.csv` from the repo; add the team's real examples to it over time.

### How to run
1. **Runtime → Change runtime type → T4 GPU → Save.** (Kaggle Notebooks with a GPU also work.)
2. **Runtime → Run all.** When asked, upload `complaint_seed_dataset.csv` (from the `ml` folder of the project).
3. Wait ~40–60 minutes. The three models are saved in **half precision (~550 MB each)** and the zip is **copied to your Google Drive** (`My Drive/resolve_models.zip`) — allow Drive access when asked. Download it from Drive straight to the E: drive.

> ⚠️ `complaint_seed_dataset.csv` is **synthetic** (written for this project) and Roman Urdu results on it are optimistic. For the final report, add real/team-written Roman Urdu complaints and re-run.

In [ ]:
# 1. Setup — Colab already has torch, transformers, pandas and scikit-learn; this only adds what's missing.
!pip -q install transformers accelerate scikit-learn sentencepiece

import torch, transformers
print("transformers", transformers.__version__, "| torch", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE — switch to a T4 GPU runtime (Runtime → Change runtime type)")

In [ ]:
# 2. Settings — change MODEL_NAME to compare models, e.g. "google/muril-base-cased" or "bert-base-multilingual-cased"
MODEL_NAME = "xlm-roberta-base"
TASKS = ["category", "urgency", "sentiment"]
EPOCHS = 8           # best epoch (validation macro-F1) is kept automatically
LEARNING_RATE = 3e-5
BATCH_SIZE = 16
MAX_LENGTH = 128     # WhatsApp complaints are short
SEED = 42
USE_BITEXT = True            # add real English support messages from Hugging Face
BITEXT_PER_CATEGORY = 300    # cap so Bitext does not drown out the Roman Urdu examples
USE_ROMAN_URDU_SENTIMENT = True   # add the public UCI Roman Urdu sentiment sentences (sentiment task only)
ROMAN_URDU_PER_CLASS = 1500       # per positive / negative / neutral
SAVE_HALF_PRECISION = True   # ~550 MB per model instead of ~1.1 GB (same predictions)
COPY_TO_DRIVE = True         # copy resolve_models.zip to Google Drive (reliable for big files)
DOWNLOAD_IN_BROWSER = False  # a ~1.6 GB browser download is slow and fills the C: drive

In [ ]:
# 3. Find the dataset CSV: Kaggle "Add Data" input, a file next to the notebook, or a Colab upload
import os, glob
found = sorted(glob.glob("/kaggle/input/**/complaint_seed_dataset*.csv", recursive=True)) +         sorted(glob.glob("complaint_seed_dataset*.csv"))
if found:
    CSV_PATH = found[0]
else:
    try:
        from google.colab import files
        uploaded = files.upload()
        CSV_PATH = next(iter(uploaded))
    except ImportError:
        raise FileNotFoundError("Add complaint_seed_dataset.csv to the notebook (Kaggle: Add Input → Upload)")
print("Using", CSV_PATH)

In [ ]:
# 4. Load and check the data
import pandas as pd

own = pd.read_csv(CSV_PATH)
missing = [c for c in ["text", "category", "language"] if c not in own.columns]
assert not missing, f"CSV is missing columns: {missing}"
for c in ["urgency", "sentiment"]:
    if c not in own.columns:
        own[c] = None
if "group_id" not in own.columns:   # each row is its own group when there are no variations
    own["group_id"] = [f"row-{i}" for i in range(len(own))]
own["source"] = own["source"] if "source" in own.columns else "uploaded_csv"
own = own.dropna(subset=["text", "category", "language"])
print(f"Uploaded CSV: {len(own)} rows")

In [ ]:
# 4b. Download the Bitext customer-support dataset (English) and map its intents to our categories
import re, random
import numpy as np
INTENT_TO_CATEGORY = {
    "get_refund": "refund_request", "track_refund": "refund_request", "check_refund_policy": "refund_request",
    "cancel_order": "cancellation", "check_cancellation_fee": "cancellation",
    "track_order": "delivery_issue", "delivery_period": "delivery_issue",
    "payment_issue": "payment_issue", "check_payment_methods": "payment_issue",
    "delivery_options": "general_inquiry", "place_order": "general_inquiry",
}   # other intents (accounts, invoices, newsletter, ...) are not post-purchase complaints and are skipped

frames = [own]
if USE_BITEXT:
    try:
        from datasets import load_dataset
        bitext = load_dataset("bitext/Bitext-customer-support-llm-chatbot-training-dataset", split="train").to_pandas()
        bitext = bitext[bitext["intent"].isin(INTENT_TO_CATEGORY)].copy()
        bitext["category"] = bitext["intent"].map(INTENT_TO_CATEGORY)
        # "{{Order Number}}" style placeholders -> plain words
        bitext["text"] = bitext["instruction"].astype(str).map(lambda t: re.sub(r"\{\{([^}]+)\}\}", lambda m: m.group(1).lower(), t))
        bitext = bitext.sample(frac=1, random_state=SEED).groupby("category").head(BITEXT_PER_CATEGORY)
        bitext = pd.DataFrame({"text": bitext["text"], "category": bitext["category"], "urgency": None, "sentiment": None,
                               "language": "en", "group_id": [f"bitext-{i}" for i in range(len(bitext))], "source": "bitext"})
        frames.append(bitext)
        print(f"Bitext: added {len(bitext)} English rows (category labels only)")
    except Exception as exc:
        print("⚠️ Could not download Bitext, continuing with the uploaded CSV only:", exc)

if USE_ROMAN_URDU_SENTIMENT:
    ru = None
    for name in ["community-datasets/roman_urdu", "roman_urdu"]:
        try:
            from datasets import load_dataset
            ru = load_dataset(name, split="train")
            print("Roman Urdu sentiment dataset:", name, len(ru), "rows")
            break
        except Exception as exc:
            print(f"  could not load {name}: {exc}")
    if ru is not None:
        feats = ru.features
        text_col = next(c for c, f in feats.items() if getattr(f, "dtype", None) == "string")
        label_col = next(c for c, f in feats.items() if hasattr(f, "names"))
        names = [n.strip().lower() for n in feats[label_col].names]
        rdf = ru.to_pandas()
        rdf["sentiment"] = rdf[label_col].map(lambda i: names[i] if isinstance(i, (int, np.integer)) and 0 <= i < len(names) else None)
        rdf["text"] = rdf[text_col].astype(str).str.strip()
        rdf = rdf[rdf["sentiment"].isin(["positive", "negative", "neutral"]) & rdf["text"].str.len().between(5, 300)]
        rdf = rdf.sample(frac=1, random_state=SEED).groupby("sentiment").head(ROMAN_URDU_PER_CLASS)
        frames.append(pd.DataFrame({"text": rdf["text"].values, "category": None, "urgency": None, "sentiment": rdf["sentiment"].values,
                                    "language": "roman_urdu", "group_id": [f"uci-{i}" for i in range(len(rdf))], "source": "uci_roman_urdu"}))
        print(f"UCI Roman Urdu: added {len(rdf)} rows (sentiment labels only):", rdf["sentiment"].value_counts().to_dict())
    else:
        print("⚠️ Roman Urdu sentiment dataset unavailable; sentiment uses the uploaded CSV only")

df = pd.concat(frames, ignore_index=True)
df["text"] = df["text"].astype(str).str.strip()
for c in ["category", "urgency", "sentiment", "language"]:
    df[c] = df[c].where(df[c].isna(), df[c].astype(str).str.strip().str.lower())
    df.loc[df[c].isin(["", "nan", "none"]), c] = None
df = df.drop_duplicates(subset=["text"]).reset_index(drop=True)

print(f"\nTotal: {len(df)} complaints, {df.group_id.nunique()} distinct groups\n")
print(pd.crosstab(df["category"], df["language"], margins=True), "\n")
for c in ["urgency", "sentiment"]:
    print(c, "labelled rows:", df[c].notna().sum()); print(df[c].value_counts().to_string(), "\n")

In [ ]:
# 5. Train / validation / test split by group, so variations of one complaint never leak into the test set
from sklearn.model_selection import GroupShuffleSplit

def group_split(frame, test_size, seed):
    gss = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=seed)
    a, b = next(gss.split(frame, groups=frame["group_id"]))
    return frame.iloc[a].reset_index(drop=True), frame.iloc[b].reset_index(drop=True)

trainval, test = group_split(df, 0.2, SEED)
train, val = group_split(trainval, 0.15, SEED)

def task_rows(frame, task):
    """Only rows that carry a label for this task (Bitext rows have no urgency/sentiment)."""
    return frame[frame[task].notna()].reset_index(drop=True)

for task in TASKS:
    tr, va, te = task_rows(train, task), task_rows(val, task), task_rows(test, task)
    print(f"{task:10s} train {len(tr):5d} | val {len(va):4d} | test {len(te):4d}")
    unseen = set(te[task]) - set(tr[task])
    if unseen:
        print(f"  ⚠️ test has labels never seen in training: {unseen} — add more examples of them")

In [ ]:
# 6. Shared evaluation helper: overall + per-language macro-F1
from sklearn.metrics import f1_score, accuracy_score, classification_report

def evaluate(y_true, y_pred, languages):
    y_true, y_pred, languages = list(y_true), list(y_pred), list(languages)
    out = {"macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
           "accuracy": accuracy_score(y_true, y_pred)}
    for lang in sorted(set(languages)):
        idx = [i for i, l in enumerate(languages) if l == lang]
        out[f"macro_f1_{lang}"] = f1_score([y_true[i] for i in idx], [y_pred[i] for i in idx], average="macro", zero_division=0)
    return out

In [ ]:
# 7. Baseline: TF-IDF (character n-grams cope with Roman Urdu spelling variations) + Logistic Regression
from sklearn.pipeline import make_pipeline, make_union
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

results = {}
for task in TASKS:
    baseline = make_pipeline(
        make_union(TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), sublinear_tf=True),
                   TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True)),
        LogisticRegression(max_iter=3000, class_weight="balanced"),
    )
    tr, te = task_rows(train, task), task_rows(test, task)
    baseline.fit(tr["text"], tr[task])
    results[("baseline", task)] = evaluate(te[task], baseline.predict(te["text"]), te["language"])
    print(task, {k: round(v, 3) for k, v in results[("baseline", task)].items()})

In [ ]:
# 8. Fine-tune the transformer, one model per task
import math, json, shutil, numpy as np, matplotlib.pyplot as plt
from datetime import date
from sklearn.metrics import ConfusionMatrixDisplay
from transformers import (AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding,
                          Trainer, TrainingArguments, set_seed)

set_seed(SEED)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
SAVE_DIR = "resolve_models"
model_tag = MODEL_NAME.split("/")[-1]

class ComplaintDataset(torch.utils.data.Dataset):
    def __init__(self, texts, labels):
        self.enc = tokenizer(list(texts), truncation=True, max_length=MAX_LENGTH)
        self.labels = list(labels)
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, i):
        item = {k: self.enc[k][i] for k in self.enc}
        item["labels"] = self.labels[i]
        return item

def compute_metrics(pred):
    preds = pred.predictions.argmax(-1)
    return {"macro_f1": f1_score(pred.label_ids, preds, average="macro", zero_division=0),
            "accuracy": accuracy_score(pred.label_ids, preds)}

def train_task(task):
    labels = sorted(df[task].dropna().unique())
    label2id = {l: i for i, l in enumerate(labels)}
    id2label = {i: l for l, i in label2id.items()}
    tr, va, te = task_rows(train, task), task_rows(val, task), task_rows(test, task)
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=len(labels), id2label=id2label, label2id=label2id)
    ds_train = ComplaintDataset(tr["text"], tr[task].map(label2id))
    ds_val = ComplaintDataset(va["text"], va[task].map(label2id))
    ds_test = ComplaintDataset(te["text"], te[task].map(label2id))
    steps = math.ceil(len(ds_train) / BATCH_SIZE) * EPOCHS
    args = TrainingArguments(
        output_dir=f"runs/{task}", eval_strategy="epoch", save_strategy="epoch",
        learning_rate=LEARNING_RATE, per_device_train_batch_size=BATCH_SIZE, per_device_eval_batch_size=64,
        num_train_epochs=EPOCHS, weight_decay=0.01, warmup_steps=max(1, int(0.1 * steps)),
        load_best_model_at_end=True, metric_for_best_model="macro_f1", greater_is_better=True,
        save_total_limit=1, fp16=torch.cuda.is_available(), report_to="none", seed=SEED, logging_steps=10,
    )
    trainer = Trainer(model=model, args=args, train_dataset=ds_train, eval_dataset=ds_val,
                      data_collator=DataCollatorWithPadding(tokenizer), compute_metrics=compute_metrics)
    trainer.train()

    pred_ids = trainer.predict(ds_test).predictions.argmax(-1)
    y_pred = [id2label[int(i)] for i in pred_ids]
    metrics = evaluate(te[task], y_pred, te["language"])
    results[(model_tag, task)] = metrics
    print(f"\n=== {task} ({MODEL_NAME}) on held-out test ===")
    print(classification_report(te[task], y_pred, zero_division=0))
    ConfusionMatrixDisplay.from_predictions(te[task], y_pred, xticks_rotation=45, cmap="Greens")
    plt.title(f"{task} — {model_tag}"); plt.tight_layout(); plt.show()

    out_dir = f"{SAVE_DIR}/{task}"
    if SAVE_HALF_PRECISION:
        trainer.model.half()   # evaluation is done; halves the file size for download and RAM
    trainer.save_model(out_dir)
    tokenizer.save_pretrained(out_dir)
    with open(f"{out_dir}/resolve_model.json", "w") as f:
        json.dump({"task": task, "base_model": MODEL_NAME, "model_version": f"{task}-{model_tag}-{date.today():%Y%m%d}",
                   "labels": labels, "test_metrics": metrics, "train_rows": len(tr),
                   "data_sources": sorted(tr["source"].dropna().unique().tolist())}, f, indent=2)
    shutil.rmtree(f"runs/{task}", ignore_errors=True)  # free disk; the best model is already saved

for task in TASKS:
    train_task(task)

In [ ]:
# 9. Results table for the report: baseline vs fine-tuned, overall and per language
summary = pd.DataFrame(results).T.round(3)
summary.index.names = ["model", "task"]
display(summary)
summary.to_csv(f"{SAVE_DIR}/results_{model_tag}.csv")

In [ ]:
# 10. Try your own messages
from transformers import pipeline
device = 0 if torch.cuda.is_available() else -1
classifiers = {t: pipeline("text-classification", model=f"{SAVE_DIR}/{t}", device=device) for t in TASKS}

examples = [
    "Mera parcel toota hua aaya, paise wapis chahiye",
    "Order kab tak aayega?",
    "I was charged twice, please fix this immediately",
    "Size chota hai, exchange ho sakta hai?",
]
for text in examples:
    labels = {t: f"{c(text)[0]['label']} ({c(text)[0]['score']:.2f})" for t, c in classifiers.items()}
    print(text, "→", labels)

In [ ]:
# 11. Package the models. Copy to Google Drive (then download from drive.google.com to the E: drive)
shutil.make_archive("resolve_models", "zip", SAVE_DIR)
size_gb = os.path.getsize("resolve_models.zip") / 1e9
print(f"resolve_models.zip: {size_gb:.2f} GB")
if os.path.isdir("/kaggle/working"):
    shutil.move("resolve_models.zip", "/kaggle/working/resolve_models.zip") if os.path.abspath(".") != "/kaggle/working" else None
    print("✅ Kaggle: download resolve_models.zip from the notebook's Output tab")
elif COPY_TO_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        shutil.copy("resolve_models.zip", "/content/drive/MyDrive/resolve_models.zip")
        print("✅ Copied to Google Drive → My Drive/resolve_models.zip")
    except Exception as exc:
        print("⚠️ Could not copy to Google Drive:", exc)
if DOWNLOAD_IN_BROWSER and not os.path.isdir("/kaggle/working"):
    from google.colab import files
    files.download("resolve_models.zip")